In [1]:
# ============================================================
# 🎬 DUBAI - AI AUTOMATED VIDEO DUBBING
# Complete Google Colab: Frontend + FastAPI Backend
# ============================================================

# =========================
# 1. INSTALL DEPENDENCIES
# =========================

!apt-get update -qq
!apt-get install -y ffmpeg -qq

!pip install -q \
    fastapi \
    uvicorn \
    gradio \
    yt-dlp \
    openai-whisper \
    deep-translator \
    edge-tts \
    requests


# =========================
# 2. IMPORTS
# =========================

import os
import asyncio
import subprocess
import threading
import time
import uuid
from pathlib import Path

import requests
import yt_dlp
import whisper
import edge_tts

from deep_translator import GoogleTranslator

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel

import uvicorn
import gradio as gr


# =========================
# 3. PROJECT DIRECTORIES
# =========================

BASE = Path("/content/DubAI")

INPUT_DIR = BASE / "input"
OUTPUT_DIR = BASE / "output"
TEMP_DIR = BASE / "temp"

for folder in [
    INPUT_DIR,
    OUTPUT_DIR,
    TEMP_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("✅ DubAI project directory created")


# ============================================================
# 4. FASTAPI BACKEND
# ============================================================

app = FastAPI(
    title="DubAI Video Dubbing API",
    version="1.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


# =========================
# JOB STORAGE
# =========================

JOBS = {}

WHISPER_MODEL = None


# =========================
# REQUEST MODEL
# =========================

class DubRequest(BaseModel):

    url: str


# =========================
# LOG FUNCTION
# =========================

def add_log(
    job_id,
    message
):

    JOBS[job_id].setdefault(
        "logs",
        []
    )

    JOBS[job_id]["logs"].append(
        message
    )

    print(
        f"[{job_id}] {message}"
    )


# =========================
# UPDATE JOB
# =========================

def update_job(
    job_id,
    **kwargs
):

    JOBS[job_id].update(
        kwargs
    )


# ============================================================
# COMMAND RUNNER
# ============================================================

def run_command(command):

    result = subprocess.run(

        command,

        stdout=subprocess.PIPE,

        stderr=subprocess.PIPE,

        text=True
    )

    if result.returncode != 0:

        raise RuntimeError(
            result.stderr[-3000:]
        )

    return result.stdout


# ============================================================
# DOWNLOAD VIDEO
# ============================================================

def download_video(
    url,
    workdir,
    job_id
):

    add_log(
        job_id,
        "⬇️ Downloading YouTube video..."
    )

    output_template = str(
        workdir /
        "source.%(ext)s"
    )

    options = {

        "outtmpl":
        output_template,

        "format":
        "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best",

        "merge_output_format":
        "mp4",

        "noplaylist":
        True,

        "quiet":
        True,

        "no_warnings":
        True
    }

    with yt_dlp.YoutubeDL(
        options
    ) as ydl:

        info = ydl.extract_info(
            url,
            download=True
        )

    files = list(
        workdir.glob(
            "source.*"
        )
    )

    video = None

    for file in files:

        if file.suffix.lower() in [

            ".mp4",
            ".mkv",
            ".webm",
            ".mov"

        ]:

            video = file
            break

    if video is None:

        raise RuntimeError(
            "Video download failed."
        )

    add_log(
        job_id,
        "✅ Video downloaded successfully"
    )

    add_log(
        job_id,
        f"🎬 Title: {info.get('title','Video')}"
    )

    return video


# ============================================================
# EXTRACT AUDIO
# ============================================================

def extract_audio(
    video,
    workdir,
    job_id
):

    add_log(
        job_id,
        "🔊 Extracting audio..."
    )

    audio = (
        workdir /
        "audio.wav"
    )

    command = [

        "ffmpeg",
        "-y",

        "-i",
        str(video),

        "-vn",

        "-ac",
        "1",

        "-ar",
        "16000",

        str(audio)
    ]

    run_command(
        command
    )

    add_log(
        job_id,
        "✅ Audio extracted"
    )

    return audio


# ============================================================
# WHISPER
# ============================================================

def load_whisper():

    global WHISPER_MODEL

    if WHISPER_MODEL is None:

        add_log(
            CURRENT_JOB,
            "🧠 Loading Whisper model..."
        )

        WHISPER_MODEL = whisper.load_model(
            "base"
        )

    return WHISPER_MODEL


# ============================================================
# TRANSCRIPTION
# ============================================================

def transcribe(
    audio,
    job_id
):

    global CURRENT_JOB

    CURRENT_JOB = job_id

    add_log(
        job_id,
        "🎙️ Transcribing with Whisper..."
    )

    model = load_whisper()

    result = model.transcribe(

        str(audio),

        fp16=False
    )

    language = result.get(
        "language",
        "unknown"
    )

    segments = []

    for segment in result.get(
        "segments",
        []
    ):

        text = segment.get(
            "text",
            ""
        ).strip()

        if text:

            segments.append({

                "start":
                float(
                    segment["start"]
                ),

                "end":
                float(
                    segment["end"]
                ),

                "text":
                text
            })

    add_log(
        job_id,
        f"🌐 Detected language: {language}"
    )

    add_log(
        job_id,
        f"📝 Segments detected: {len(segments)}"
    )

    return language, segments


# ============================================================
# TRANSLATION
# ============================================================

def translate_segments(
    segments,
    job_id
):

    add_log(
        job_id,
        "🌍 Translating speech to English..."
    )

    translator = GoogleTranslator(
        source="auto",
        target="en"
    )

    result = []

    total = len(
        segments
    )

    for index, segment in enumerate(
        segments
    ):

        try:

            english = translator.translate(
                segment["text"]
            )

        except Exception:

            english = segment["text"]

        result.append({

            "start":
            segment["start"],

            "end":
            segment["end"],

            "text":
            segment["text"],

            "translation":
            english
        })

        progress = (
            30 +
            int(
                20 *
                (index + 1) /
                max(total, 1)
            )
        )

        update_job(
            job_id,
            progress=progress
        )

    add_log(
        job_id,
        "✅ English translation completed"
    )

    return result


# ============================================================
# EDGE TTS
# ============================================================

async def create_voice(
    text,
    output
):

    communicate = edge_tts.Communicate(

        text,

        "en-US-AriaNeural"
    )

    await communicate.save(
        str(output)
    )


# ============================================================
# CREATE DUBBED AUDIO
# ============================================================

def create_dubbed_audio(
    segments,
    workdir,
    job_id
):

    add_log(
        job_id,
        "🎤 Generating natural English voice..."
    )

    generated = []

    total = len(
        segments
    )

    for index, segment in enumerate(
        segments
    ):

        output = (
            workdir /
            f"speech_{index}.mp3"
        )

        asyncio.run(
            create_voice(
                segment["translation"],
                output
            )
        )

        generated.append(
            (
                segment,
                output
            )
        )

        progress = (
            50 +
            int(
                30 *
                (index + 1) /
                max(total, 1)
            )
        )

        update_job(
            job_id,
            progress=progress
        )

    # ---------------------------
    # BUILD FFMPEG FILTER
    # ---------------------------

    inputs = []

    filters = []

    for index, (
        segment,
        audio
    ) in enumerate(generated):

        inputs += [
            "-i",
            str(audio)
        ]

        delay = int(
            segment["start"] *
            1000
        )

        filters.append(

            f"[{index}:a]"
            f"adelay={delay}|{delay}"
            f"[a{index}]"
        )

    labels = ""

    for index in range(
        len(generated)
    ):

        labels += (
            f"[a{index}]"
        )

    filter_complex = (

        ";".join(filters)

        + ";"

        + labels

        + f"amix="
          f"inputs={len(generated)}:"
          f"duration=longest:"
          f"normalize=0"
          f"[out]"
    )

    output_audio = (
        workdir /
        "english_audio.m4a"
    )

    command = [

        "ffmpeg",
        "-y",

        *inputs,

        "-filter_complex",
        filter_complex,

        "-map",
        "[out]",

        "-c:a",
        "aac",

        "-b:a",
        "192k",

        str(output_audio)
    ]

    run_command(
        command
    )

    add_log(
        job_id,
        "✅ English audio generated"
    )

    return output_audio


# ============================================================
# FINAL VIDEO
# ============================================================

def create_final_video(
    video,
    english_audio,
    job_id
):

    add_log(
        job_id,
        "🎬 Replacing original audio..."
    )

    output = (
        OUTPUT_DIR /
        f"dubbed_{job_id}.mp4"
    )

    command = [

        "ffmpeg",
        "-y",

        "-i",
        str(video),

        "-i",
        str(english_audio),

        "-map",
        "0:v:0",

        "-map",
        "1:a:0",

        "-c:v",
        "copy",

        "-c:a",
        "aac",

        "-b:a",
        "192k",

        "-shortest",

        str(output)
    ]

    run_command(
        command
    )

    add_log(
        job_id,
        "✅ Final dubbed video created!"
    )

    return output


# ============================================================
# COMPLETE PIPELINE
# ============================================================

def process_job(
    job_id,
    url
):

    start = time.time()

    workdir = (
        TEMP_DIR /
        job_id
    )

    workdir.mkdir(
        parents=True,
        exist_ok=True
    )

    try:

        update_job(
            job_id,
            status="processing",
            progress=5
        )

        # 1 DOWNLOAD

        video = download_video(
            url,
            workdir,
            job_id
        )

        update_job(
            job_id,
            progress=15
        )

        # 2 AUDIO

        audio = extract_audio(
            video,
            workdir,
            job_id
        )

        update_job(
            job_id,
            progress=20
        )

        # 3 WHISPER

        language, segments = (
            transcribe(
                audio,
                job_id
            )
        )

        update_job(
            job_id,
            progress=30,
            language=language
        )

        # 4 TRANSLATION

        translated = (
            translate_segments(
                segments,
                job_id
            )
        )

        # 5 TTS

        english_audio = (
            create_dubbed_audio(
                translated,
                workdir,
                job_id
            )
        )

        update_job(
            job_id,
            progress=85
        )

        # 6 FINAL VIDEO

        output = create_final_video(
            video,
            english_audio,
            job_id
        )

        elapsed = round(
            time.time() - start,
            2
        )

        update_job(

            job_id,

            status="completed",

            progress=100,

            output=str(output),

            processing_time=elapsed
        )

        add_log(
            job_id,
            f"⏱️ Processing time: {elapsed} seconds"
        )

    except Exception as e:

        update_job(

            job_id,

            status="failed",

            progress=0,

            error=str(e)
        )

        add_log(
            job_id,
            f"❌ ERROR: {e}"
        )


# ============================================================
# FASTAPI ROUTES
# ============================================================

@app.get("/api/health")
def health():

    return {
        "status": "online"
    }


@app.post("/api/dub")
def start_dubbing(
    request: DubRequest
):

    if not request.url.startswith(
        "http"
    ):

        raise HTTPException(
            400,
            "Invalid YouTube URL"
        )

    job_id = uuid.uuid4().hex[:8]

    JOBS[job_id] = {

        "status":
        "queued",

        "progress":
        0,

        "logs":
        ["🚀 Job created"]
    }

    thread = threading.Thread(

        target=process_job,

        args=(
            job_id,
            request.url
        ),

        daemon=True
    )

    thread.start()

    return {
        "job_id":
        job_id
    }


@app.get(
    "/api/status/{job_id}"
)
def get_status(
    job_id
):

    if job_id not in JOBS:

        raise HTTPException(
            404,
            "Job not found"
        )

    return JOBS[job_id]


# ============================================================
# START FASTAPI
# ============================================================

def start_backend():

    uvicorn.run(

        app,

        host="127.0.0.1",

        port=8000,

        log_level="warning"
    )


backend_thread = threading.Thread(

    target=start_backend,

    daemon=True
)

backend_thread.start()

time.sleep(3)

print(
    "✅ FastAPI backend running on port 8000"
)


# ============================================================
# 5. FRONTEND
# ============================================================

def run_dubbing(url):

    if not url:

        yield (
            0,
            "❌ Please enter a YouTube URL.",
            None,
            "Waiting..."
        )

        return


    # -------------------------
    # START BACKEND JOB
    # -------------------------

    try:

        response = requests.post(

            "http://127.0.0.1:8000/api/dub",

            json={
                "url": url
            },

            timeout=30
        )

        data = response.json()

        job_id = data[
            "job_id"
        ]

    except Exception as e:

        yield (
            0,
            f"❌ Backend error: {e}",
            None,
            "Backend error"
        )

        return


    # -------------------------
    # POLL STATUS
    # -------------------------

    while True:

        try:

            status = requests.get(

                f"http://127.0.0.1:8000/api/status/{job_id}",

                timeout=30

            ).json()

        except Exception as e:

            yield (
                0,
                str(e),
                None,
                "Connection error"
            )

            return


        progress = status.get(
            "progress",
            0
        )

        logs = status.get(
            "logs",
            []
        )

        log_text = "\n".join(
            logs
        )

        current_status = status.get(
            "status",
            "processing"
        )


        # COMPLETED

        if current_status == "completed":

            output = status[
                "output"
            ]

            yield (

                100,

                log_text,

                output,

                "🎉 Dubbing completed!"
            )

            break


        # FAILED

        if current_status == "failed":

            yield (

                0,

                log_text,

                None,

                "❌ Processing failed"
            )

            break


        # PROCESSING

        yield (

            progress,

            log_text,

            None,

            f"⚙️ Processing... {progress}%"
        )

        time.sleep(2)


# ============================================================
# FRONTEND CSS
# ============================================================

css = """

body {

    background:
    linear-gradient(
        135deg,
        #020617,
        #0f172a,
        #111827
    );

}


.gradio-container {

    max-width: 1200px !important;

    margin: auto !important;

}


#title {

    text-align: center;

    padding: 25px;

}


#title h1 {

    font-size: 45px;

    background:
    linear-gradient(
        90deg,
        #a855f7,
        #3b82f6
    );

    -webkit-background-clip: text;

    color: transparent;

}


.card {

    border-radius: 18px !important;

}


button {

    border-radius: 12px !important;

}


"""

# ============================================================
# BUILD FRONTEND
# ============================================================

with gr.Blocks(
    title="DubAI - AI Video Dubbing",
    css=css,
    theme=gr.themes.Soft(
        primary_hue="indigo"
    )
) as frontend:


    # -------------------------
    # HEADER
    # -------------------------

    gr.HTML(
        """
        <div id="title">

        <h1>🎬 DubAI</h1>

        <h2>
        AI Video Dubbing
        </h2>

        <p>
        Convert YouTube videos from any language
        into natural English-dubbed videos.
        </p>

        </div>
        """
    )


    # -------------------------
    # FEATURES
    # -------------------------

    with gr.Row():

        gr.Markdown(
            """
            ⚡ **Multi-language**

            Hindi, French, German,
            Spanish and more.
            """
        )

        gr.Markdown(
            """
            🎙️ **Whisper**

            Accurate speech
            transcription.
            """
        )

        gr.Markdown(
            """
            🌍 **Translation**

            AI-powered English
            translation.
            """
        )

        gr.Markdown(
            """
            🔊 **Natural Voice**

            Edge TTS English
            voice generation.
            """
        )


    # -------------------------
    # URL INPUT
    # -------------------------

    with gr.Group(
        elem_classes="card"
    ):

        gr.Markdown(
            """
            ## 🔗 1. Enter YouTube Video

            Paste the YouTube URL
            that you want to dub.
            """
        )

        url = gr.Textbox(

            label="YouTube URL",

            placeholder=
            "https://www.youtube.com/watch?v=example"
        )


    # -------------------------
    # BUTTON
    # -------------------------

    start_button = gr.Button(

        "🚀 START DUBBING",

        variant="primary",

        size="lg"
    )


    # -------------------------
    # PROGRESS
    # -------------------------

    with gr.Group(
        elem_classes="card"
    ):

        gr.Markdown(
            """
            ## ⚙️ 2. Processing Progress
            """
        )

        progress = gr.Slider(

            minimum=0,

            maximum=100,

            value=0,

            label="Processing"
        )


        current_status = gr.Markdown(
            "Waiting for video..."
        )


    # -------------------------
    # LOGS
    # -------------------------

    with gr.Group(
        elem_classes="card"
    ):

        gr.Markdown(
            """
            ## 📋 Processing Logs
            """
        )

        logs = gr.Textbox(

            lines=15,

            label="Live Logs",

            interactive=False
        )


    # -------------------------
    # RESULT
    # -------------------------

    with gr.Group(
        elem_classes="card"
    ):

        gr.Markdown(
            """
            ## 🎥 3. Final Dubbed Video
            """
        )

        output_video = gr.Video(

            label=
            "English Dubbed Video",

            interactive=False
        )


    # -------------------------
    # RUN
    # -------------------------

    start_button.click(

        fn=run_dubbing,

        inputs=url,

        outputs=[

            progress,

            logs,

            output_video,

            current_status

        ]
    )


# ============================================================
# LAUNCH FRONTEND
# ============================================================

frontend.launch(
    share=True,
    debug=False
)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 20.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 4.4 MB/s eta 0:00:00
✅ DubAI project directory created
✅ FastAPI backend running on port 8000


/tmp/ipykernel_603/3751979985.py:1203: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://adc712d390662f1750.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
